In [67]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


EDA visualizations: yield trends over time, bryophyte species richness and identification counts by year, correlation matrix across all variables, and species richness per site versus yield scatter.

In [68]:

DATA_DIR = "preprocessing_results"
DATA_FULL_FILE = "EDA_merged_ABC_2010_2019.csv"
DATA_WHEAT_FILE = "wheat_yield_full_historical.csv"

df = pd.read_csv(os.path.join(DATA_DIR, DATA_FULL_FILE))

df = df.sort_values("Year").reset_index(drop=True)

wheat_full = pd.read_csv(os.path.join(DATA_DIR, DATA_WHEAT_FILE)) # used for yield trend plot for better context
wheat_full = wheat_full.sort_values("Year").reset_index(drop=True)


### plot_time_series()

Display line plots of the given data.

In [69]:
def plot_time_series(df, columns, title):
    plt.figure()

    for col in columns:
        plt.plot(df["Year"], df[col], marker="o", label=col)

    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.title(title)
    plt.legend()
    plt.grid()

    plt.show()

In [ ]:
# Used to reveal interactions
df['temp_richness_interaction'] = df['mean_summer_temp_C'] * df['richness_per_site']
    
plot_time_series(wheat_full, ["wheat_yield_bu_acre"], "Historical Wheat Yield (Full Series)")

plot_time_series(df, ["richness_per_site", "ids_per_site"], "Bryophyte Indicators Over Time")

plot_time_series(
    df,
        [
            "mean_summer_temp_C",
            "total_growing_precip_mm",
            "frost_free_days",
            "spring_temp_anomaly_C"
        ],
        "Climate Variables Over Time"
)


### plot_correlation_matrix()

Shows the correlation matrix as a heat map, blue for negative correlations, yellow for positive.

It's simple so it can be used pretty universally.

In [ ]:
def plot_correlation_matrix(df, cols=None):
    if cols is not None:
        data = df[cols]
    else:
        data = df

    corr = data.corr(numeric_only=True)

    plt.figure()
    plt.imshow(corr)
    plt.colorbar()

    labels = corr.columns
    plt.xticks(range(len(labels)), labels)
    plt.yticks(range(len(labels)), labels)

    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()

In [ ]:

plot_correlation_matrix(df)

plot_correlation_matrix(
    df,
    [
        "wheat_yield_bu_acre",
        "richness_per_site",
        "ids_per_site",
        "mean_summer_temp_C",
        "total_growing_precip_mm",
        "frost_free_days",
        "spring_temp_anomaly_C"
    ]
)

### scatter_set()

Displays scatter set for given input.

It's simple so it can be used pretty universally.

In [71]:
def scatter_set(df, x_cols, y_col):
    for x_col in x_cols:
        x = df[x_col]
        y = df[y_col]

        plt.figure()
        plt.scatter(x, y)

        if len(x) > 1:
            m, b = np.polyfit(x, y, 1)
            plt.plot(x, m * x + b)

        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.title(f"{x_col} vs {y_col}")

        plt.grid()
        plt.show()

In [ ]:

scatter_set(
    df,
    ["richness_per_site", "ids_per_site"],
    "wheat_yield_bu_acre"
)

scatter_set(
    df,
    [
        "mean_summer_temp_C",
        "total_growing_precip_mm",
        "frost_free_days",
        "spring_temp_anomaly_C"
    ],
    "wheat_yield_bu_acre"
)



### Run_modeling_diagnostics()

Checks multicollinearity using variance inflation factor (VIF), and influences using cook's distance.

Displays the results in an easy to interperate manner

In [72]:

def run_modeling_diagnostics(df, feature_cols, target_col):
    data = df[feature_cols + [target_col]].dropna()
    X = data[feature_cols]
    y = data[target_col]
    X_const = sm.add_constant(X)
    
    # Multicollinearity Check
    vif_df = pd.DataFrame()
    vif_df["feature"] = X.columns
    vif_df["VIF"] = [variance_inflation_factor(X_const.values, i+1) for i in range(len(X.columns))]
    
    # Influence Check (Cook's Distance)
    model = sm.OLS(y, X_const).fit()
    influence = model.get_influence()
    cooks_d = influence.cooks_distance[0]
    
    print(f"\n{target_col}")
    print("\nVIF Scores:")
    print(vif_df)
    
    print("\nCook's Distance (by row index):")
    for i, d in enumerate(cooks_d):
        label = "!!!" if d > (4/len(X)) else ""
        print(f"Index {i}: {d:.4f} {label}")

    print("\nModel Coefficients:")
    print(model.params[1:])
    return model

In [ ]:
model_b_features = [
        "richness_per_site", 
        "mean_summer_temp_C",
        "spring_temp_anomaly_C", 
        "total_growing_precip_mm",
]

model_a_features = [
    "mean_summer_temp_C", 
    "total_growing_precip_mm", 
    "spring_temp_anomaly_C"
]

model_c_features = [
        "richness_per_site",  
]
    
run_modeling_diagnostics(df, model_a_features, "wheat_yield_bu_acre")
run_modeling_diagnostics(df, model_b_features, "wheat_yield_bu_acre")
run_modeling_diagnostics(df, model_c_features, "wheat_yield_bu_acre")

Model A uses:
* Mean Summer Temperature
* Total Growing Season Precipitation
* Spring Temperature Anomaly  
 
 This is because the correlation matrix and VIF diagnostics proved these are the most robust, non-redundant climate drivers, specifically highlighting the positive impact of warm springs versus the negative impact of summer heat stress. 
 
Model B uses:
* Mean Summer Temperature
* Total Growing Season Precipitation
* Spring Temperature Anomaly  
* Richness per Site  

This is because exploratory diagnostics showed that adding this biological indicator increases the model explanatory power and provides a unique ground level signal that is not captured by meteorological data alone. 

Model C uses only Richness Per Site because it serves as a pure biological proxy to test the standalone relationship between biodiversity and yield, and it was chosen over total identifications to prioritize ecological diversity as a more stable indicator of microclimate conditions. Together, these models allow for a clear comparison between traditional climate modeling and integrated biological forecasting.